**Creating DataFrames, Unique Records & Dropping Duplicates**

**Creating DataFrames from a List**

Method 1 — createDataFrame() with a list of tuples

Method 2 — createDataFrame() with explicit schema

Method 3 — From a CSV file

Method 4 — From an RDD


**Finding Unique Records — distinct()**

distinct() returns a new DataFrame with duplicate rows removed. 

It considers ALL columns when determining uniqueness — a row is a duplicate only if every column value is identical to another row.


**Dropping Duplicates — dropDuplicates()**

dropDuplicates() is more powerful than distinct() — it lets you define which columns to consider when identifying duplicates. The first occurrence is kept, the rest are dropped.

**distinct() vs dropDuplicates():**

 distinct() always considers all columns. dropDuplicates() lets you specify which columns to consider — this is the key difference. In production, dropDuplicates(["id_column"]) is the standard approach for deduplicating on a primary key.

In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-8")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-088e0b37-5373-4936-880c-17bbecf1a7ac;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 152ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Create a DataFrame from scratch using createDataFrame() with an explicit schema. Use this data — 5 rows with columns: employee_id (String), name (String), department (String), salary (Double). Include at least one None value. Show the DataFrame and print the schema.

In [2]:
from pyspark.sql.types import *
schema=StructType([
    StructField("employee_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
])
data=[("1", "John Doe", "Engineering", 75000.0),
      ("2", "Jane Smith", "Marketing", 65000.0),
      ("3", "Alice Johnson", "Sales", 70000.0),
      ("4", "Bob Brown", "Engineering", 80000.0),
      ("5", "Charlie Davis", None, 60000.0)]
df=spark.createDataFrame(data, schema)
df.show()
df.printSchema()


+-----------+-------------+-----------+-------+
|employee_id|         name| department| salary|
+-----------+-------------+-----------+-------+
|          1|     John Doe|Engineering|75000.0|
|          2|   Jane Smith|  Marketing|65000.0|
|          3|Alice Johnson|      Sales|70000.0|
|          4|    Bob Brown|Engineering|80000.0|
|          5|Charlie Davis|       NULL|60000.0|
+-----------+-------------+-----------+-------+

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: double (nullable = true)



**Task 2**

From orders.csv, find all unique values of payment_method using distinct(). Then find all unique combinations of region + status. How many unique combinations are there?

In [3]:
orders=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv", header=True,inferSchema=True)
print(f"Distinct payment methods: {orders.select('payment_method').distinct().count()}")
print(f"Distinct region-status combinations: {orders.select('region', 'status').distinct().count()}")

26/08/09 07:34:10 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Distinct payment methods: 3


Distinct region-status combinations: 15


**Task 3**

Create a DataFrame with intentional duplicates — repeat at least 2 rows exactly. Use dropDuplicates() with no arguments to remove exact duplicates. Print the row count before and after.

In [4]:
data=[("1", "John Doe", "Engineering", 75000.0),
      ("1", "John Doe", "Engineering", 75000.0),
      ("2", "Jane Smith", "Marketing", 65000.0),
      ("3", "Alice Johnson", "Sales", 70000.0),
      ("2", "Jane Smith", "Marketing", 65000.0),
      ("5", "Charlie Davis", None, 60000.0)]
df=spark.createDataFrame(data, schema)
print(f"Total rows: {df.count()}")
print(f"Rows after dropping duplicates: {df.dropDuplicates().count()}")

Total rows: 6
Rows after dropping duplicates: 4


**Task 4**

From orders.csv, use dropDuplicates(["customer_id"]) to keep only the first order per customer. How many rows remain? What does this tell you about the dataset?

In [5]:
orders_df=spark.read.csv("s3a://pyspark-30-days-rahul-2026/data/orders.csv",
header=True,inferSchema=True)
print(f"Total rows: {orders_df.count()}")
print(f"Rows after dropping duplicates: {orders_df.dropDuplicates(['customer_id']).count()}")

Total rows: 100
Rows after dropping duplicates: 25


In [6]:
spark.stop()